In [337]:
# This notebook is a work in progress
_cf_value_cache = {}

def cf_value(terms):
    # return continued_fraction of terms as a real number (cached)
    key = tuple(tuple(part) for part in terms)
    if key not in _cf_value_cache:
        _cf_value_cache[key] = continued_fraction(terms).n()
    return _cf_value_cache[key]

# fixed eta endpoints used in H (cached)
_ETA_1 = None
_ETA_2 = None

def _etas():
    global _ETA_1, _ETA_2
    if _ETA_1 is None:
        _ETA_1 = cf_value([(0,), (1, 3)])
        _ETA_2 = cf_value([(0,), (3, 1)])
    return _ETA_1, _ETA_2

def cf_extrema(term_lists):
    # return (min_value, min_terms), (max_value, max_terms) for a list of CF term lists
    valued = [(cf_value(terms), terms) for terms in term_lists]
    return (
        min(valued, key=lambda item: item[0]),
        max(valued, key=lambda item: item[0]),
    )

def sr_extrema(sr_candidates, component):
    # component: 0 for s, 1 for r; sr_candidates is e.g. sr_1[j] or sr_2[j]
    return cf_extrema(pair[component] for pair in sr_candidates)

def check_ii(h, k, sr_begin, sr_end):
    # verify connectivity condition (ii) between sr_1[0] and sr_2[-1]
    (s_lo, _), (s_hi, _) = sr_extrema(sr_begin, 0)
    (r_lo, _), (r_hi, _) = sr_extrema(sr_begin, 1)
    (s2_lo, _), (s2_hi, _) = sr_extrema(sr_end, 0)
    (r2_lo, _), (r2_hi, _) = sr_extrema(sr_end, 1)

    s_ok = (s_hi <= s2_lo) if h % 2 == 0 else (s_lo >= s2_hi)
    r_ok = (r_hi <= r2_lo) if k % 2 == 0 else (r_lo >= r2_hi)
    return {"s": s_ok, "r": r_ok}

def G(r, s, t, v, xv=None):
    # xv=None returns a lightweight numeric callable; otherwise numeric value.
    # Pure arithmetic — no symbolic var/subs (symbolic path was the bottleneck).
    r, s, t, v = float(r), float(s), float(t), float(v)
    def Gx(x):
        x = float(x)
        return (1.0 + r * x) * (1.0 + s * x) / ((1.0 + t * x) * (1.0 + v * x))
    return Gx if xv is None else Gx(xv)

def H(s_1, s_2, r_2, r_1, c_plus, c_minus):
    eta_1, eta_2 = _etas()
    return (
        ((s_1 - s_2) / (r_2 - r_1))
        * G(r_2, r_1, eta_1, eta_2, c_plus)
        * G(eta_1, eta_2, s_1, s_2, c_minus)
    )

def _g_extrema_on_interval(r, s, t, v, x_range):
    """Extrema of G(r,s,t,v; x) on [lo, hi] by endpoints + analytic critical points.
    G = (1+(r+s)x+rs x^2)/(1+(t+v)x+tv x^2); G'=0 reduces to a quadratic.
    """
    lo, hi = float(x_range[0]), float(x_range[1])
    r, s, t, v = float(r), float(s), float(t), float(v)
    if r == t and s == v:
        g = G(r, s, t, v, lo)
        return (g, lo), (g, hi)

    a0, a1, a2 = 1.0, r + s, r * s
    b0, b1, b2 = 1.0, t + v, t * v
    # N'D - ND' = A x^2 + B x + C  (x^3 cancels)
    A = a2 * b1 - a1 * b2
    B = 2.0 * (a2 * b0 - a0 * b2)
    C = a1 * b0 - a0 * b1

    xs = [lo, hi]
    eps = 1e-14
    if abs(A) < eps:
        if abs(B) >= eps:
            x0 = -C / B
            if lo <= x0 <= hi:
                xs.append(x0)
    else:
        disc = B * B - 4.0 * A * C
        if disc >= 0:
            sqrt_disc = disc ** 0.5
            for x0 in ((-B + sqrt_disc) / (2.0 * A), (-B - sqrt_disc) / (2.0 * A)):
                if lo <= x0 <= hi:
                    xs.append(x0)

    valued = [(G(r, s, t, v, x0), x0) for x0 in xs]
    return min(valued, key=lambda item: item[0]), max(valued, key=lambda item: item[0])

_min_max_H_cache = {}

def min_max_H(s_1, s_2, r_2, r_1, c_minus_range=(0.2, 0.9), c_plus_range=(0.2, 0.9)):
    key = (
        float(s_1), float(s_2), float(r_2), float(r_1),
        float(c_minus_range[0]), float(c_minus_range[1]),
        float(c_plus_range[0]), float(c_plus_range[1]),
    )
    if key in _min_max_H_cache:
        return _min_max_H_cache[key]

    eta_1, eta_2 = _etas()
    g1 = _g_extrema_on_interval(r_2, r_1, eta_1, eta_2, c_plus_range)
    g2 = _g_extrema_on_interval(eta_1, eta_2, s_1, s_2, c_minus_range)
    candidates = [
        (H(s_1, s_2, r_2, r_1, c_plus, c_minus), c_plus, c_minus)
        for _, c_plus in g1
        for _, c_minus in g2
    ]
    result = (
        min(candidates, key=lambda item: item[0]),
        max(candidates, key=lambda item: item[0]),
    )
    _min_max_H_cache[key] = result
    return result

def _cf_candidates(side, family):
    extensions = ((1, 3), (3, 1)) if family == "eta" else ((1, 2), (2, 1))
    return [(cf_value([(0, *side), ext]), ext) for ext in extensions]

def _pick_cf(side, family, which):
    pairs = _cf_candidates(side, family)
    return (min if which == "min" else max)(pairs, key=lambda item: item[0])

def _t_interval_expression(negative_side, neg_ext, positive_side, pos_ext):
    return (0, negative_side, neg_ext, positive_side, pos_ext)

def T_interval (negative_side, positive_side):
    # return beginning/end of T_interval I(negative_side | positive_side),
    # together with expressions witnessing each endpoint

    beginning_candidates = []
    end_candidates = []

    for pos_family, neg_family in [("eta", "theta"), ("theta", "eta")]:
        pos_b, pos_ext_b = _pick_cf(positive_side, pos_family, "min")
        neg_b, neg_ext_b = _pick_cf(negative_side, neg_family, "min")
        beginning_candidates.append((
            pos_b + neg_b,
            _t_interval_expression(negative_side, neg_ext_b, positive_side, pos_ext_b),
        ))

        pos_e, pos_ext_e = _pick_cf(positive_side, pos_family, "max")
        neg_e, neg_ext_e = _pick_cf(negative_side, neg_family, "max")
        end_candidates.append((
            pos_e + neg_e,
            _t_interval_expression(negative_side, neg_ext_e, positive_side, pos_ext_e),
        ))

    beginning, beginning_expr = min(beginning_candidates, key=lambda item: item[0])
    end, end_expr = max(end_candidates, key=lambda item: item[0])
    return beginning, beginning_expr, end, end_expr

def T_ratio (negative_side, positive_side):
    # return T_ratio of I(negative_side | positive_side)
    positive_side_eta = [cf_value([(0, *positive_side), (1,3)]), cf_value([(0, *positive_side), (3,1)])]
    negative_side_eta = [cf_value([(0, *negative_side), (1,3)]), cf_value([(0, *negative_side), (3,1)])]

    return abs( (positive_side_eta[0] - positive_side_eta[1]) / (negative_side_eta[0] - negative_side_eta[1]) )

_ETA_EXTS = ((3, 1), (1, 3))
_THETA_EXTS = ((2, 1), (1, 2))
_DEFAULT_V_A_LO = RR(1)
_DEFAULT_V_A_HI = RR(17) / 5

def _pair_H_margins(cand_a, cand_b, v_lo, v_hi, h, k):
    """One min_max_H call -> (margin_a_b, margin_b_a).
    margin >= 0 means (-1)^{h+k} I(first) <= (-1)^{h+k} I(second) on the whole regime.
    Returns (None, None) if r values coincide (H undefined).
    """
    s1, r1 = cand_a
    s2, r2 = cand_b
    r1_val, r2_val = cf_value(r1), cf_value(r2)
    delta = r2_val - r1_val
    if delta == 0:
        return None, None
    s1_val, s2_val = cf_value(s1), cf_value(s2)
    (min_H, _, _), (max_H, _, _) = min_max_H(s1_val, s2_val, r2_val, r1_val)
    sign_h = (-1) ** h
    sign_k = (-1) ** k
    f_vals = [
        delta * (sign_h * float(v) - sign_k * float(H))
        for v in (v_lo, v_hi)
        for H in (min_H, max_H)
    ]
    return min(f_vals), min(-f for f in f_vals)

def _select_by_H_dominance(candidates, mode, v_lo, v_hi, h, k, tol=1e-12):
    """Compare raw T-interval endpoints I via H-margins.

    _pair_H_margins gives margins for the *signed* order
        (-1)^{h+k} I(a)  ?  (-1)^{h+k} I(b).
    We convert that to the raw order I(a) ? I(b): when h+k is odd the
    inequality flips, because (-1)^{h+k} = -1.

    mode 'min': keep the always-smaller raw-I candidate (sr_1 / left end).
    mode 'max': keep the always-larger raw-I candidate (sr_2 / right end).
    On incomparability or mutual equality, keep all candidates.
    """
    assert len(candidates) == 2
    a, b = candidates[0], candidates[1]
    margin_a_b, margin_b_a = _pair_H_margins(a, b, v_lo, v_hi, h, k)
    sigma = (-1) ** (h + k)
    if margin_a_b is None:
        a_leq_b = b_leq_a = False
    elif sigma > 0:
        # signed order == raw order
        a_leq_b = margin_a_b >= -tol
        b_leq_a = margin_b_a >= -tol
    else:
        # signed order is opposite raw order: signed(a)<=signed(b) <=> I(a)>=I(b)
        a_leq_b = margin_b_a >= -tol
        b_leq_a = margin_a_b >= -tol
    info = {
        "a_leq_b": a_leq_b,
        "b_leq_a": b_leq_a,
        "margin_a_b": margin_a_b,
        "margin_b_a": margin_b_a,
        "sigma": sigma,
    }
    if mode == "min":
        if a_leq_b and not b_leq_a:
            return (a,), info
        if b_leq_a and not a_leq_b:
            return (b,), info
        return candidates, info
    if mode == "max":
        if a_leq_b and not b_leq_a:
            return (b,), info
        if b_leq_a and not a_leq_b:
            return (a,), info
        return candidates, info
    raise ValueError(mode)

def _sr_pairing_candidates(successor, h, k, parity_offset=0):
    """Two (eta/theta) pairings for T-interval beginning (0) or end (1)."""
    return (
        (
            [(0, *successor[0]), _ETA_EXTS[(len(successor[0]) + h + parity_offset) % 2]],
            [(0, *successor[1]), _THETA_EXTS[(len(successor[1]) + k + parity_offset) % 2]],
        ),
        (
            [(0, *successor[0]), _THETA_EXTS[(len(successor[0]) + h + parity_offset) % 2]],
            [(0, *successor[1]), _ETA_EXTS[(len(successor[1]) + k + parity_offset) % 2]],
        ),
    )

def successor_sr(successors, h, k, v_lo=None, v_hi=None):
    """Build sr_1 / sr_2 candidate lists from successors.

    Each sr_1[j] / sr_2[j] is a tuple of (s_terms, r_terms) pairs, filtered by
    H-dominance over v(A) in [v_lo, v_hi]. Also returns per-successor selection info.
    """
    if v_lo is None:
        v_lo = _DEFAULT_V_A_LO
    if v_hi is None:
        v_hi = _DEFAULT_V_A_HI

    sr_1, sr_2, details = [], [], []
    for j, successor in enumerate(successors):
        sr1_cands = _sr_pairing_candidates(successor, h, k, parity_offset=0)
        sr1_kept, sr1_info = _select_by_H_dominance(
            sr1_cands, "min", v_lo, v_hi, h, k
        )
        sr_1.append(sr1_kept)

        sr2_cands = _sr_pairing_candidates(successor, h, k, parity_offset=1)
        sr2_kept, sr2_info = _select_by_H_dominance(
            sr2_cands, "max", v_lo, v_hi, h, k
        )
        sr_2.append(sr2_kept)

        details.append({
            "j": j,
            "successor": successor,
            "sr_1": sr1_kept,
            "sr_1_info": sr1_info,
            "sr_2": sr2_kept,
            "sr_2_info": sr2_info,
        })
    return sr_1, sr_2, details

def _x_bounds_from_H_range(h, k, r_diff, min_H, max_H):
    """Return (x_lower, x_upper) for condition (iii); each may be None if unconstrained."""
    if r_diff == 0:
        return None, None
    sign_h = (-1) ** h
    sigma = (-1) ** (h + k)
    if sign_h * r_diff > 0:
        if sigma > 0:
            return max_H, None
        return None, -max_H
        
    if sigma > 0:
        return None, min_H
    return -min_H, None

def _iii_constraint_string(x_lo, x_hi):
    if x_lo is not None and x_hi is None:
        return f"x >= {x_lo}"
    if x_lo is None and x_hi is not None:
        return f"x <= {x_hi}"
    if x_lo is None and x_hi is None:
        return "no restriction"
    return f"{x_lo} <= x <= {x_hi}"


def _intersect_bound(a, b, how):
    """Intersect two range endpoints; None means unconstrained.
    how='lower' -> max of the two; how='upper' -> min of the two.
    """
    if a is None:
        return b
    if b is None:
        return a
    return max(a, b) if how == "lower" else min(a, b)


def iii_v_range(successors, h, k, sr_1=None, sr_2=None, v_lo=None, v_hi=None):
    """Return the v(A) range forced by condition (iii) across successor pairs.

    Compares sr_1[j] with sr_2[j-1] for j = 1, ..., len(successors)-1.
    If sr_1 / sr_2 are omitted, they are built via successor_sr.
    """
    
    # Known inequality for each (s_1, s_2, r_1, r_2):
    #   (-1)^h * x * (r_2 - r_1) >= (-1)^k * (r_2 - r_1) * H(s_1, s_2, r_2, r_1, c_plus, c_minus)
    # Equivalent form (when r_2 != r_1):
    #   (r_2 - r_1) * (-1)^h * (x - (-1)^{h+k} * H) >= 0
    # So with sigma = (-1)^{h+k}:
    #   if (r_2 - r_1) * (-1)^h > 0:  x >= sigma * H
    #   if (r_2 - r_1) * (-1)^h < 0:  x <= sigma * H
    # H varies in [min_H, max_H] as c_plus, c_minus vary; we take the bound
    # forced for every such H (worst-case over the H-range).

    if sr_1 is None or sr_2 is None:
        sr_1, sr_2, _ = successor_sr(successors, h, k, v_lo, v_hi)

    sign_h = (-1) ** h
    sign_k = (-1) ** k
    sigma = sign_h * sign_k
    by_j = {}

    for j in range(1, len(successors)):
        records = []
        for s_1, r_1 in sr_1[j]:
            for s_2, r_2 in sr_2[j - 1]:
                r_1_val = cf_value(r_1)
                r_2_val = cf_value(r_2)
                r_diff = r_2_val - r_1_val
                (min_H, _, _), (max_H, _, _) = min_max_H(
                    cf_value(s_1), cf_value(s_2), r_2_val, r_1_val
                )
                x_lo, x_hi = _x_bounds_from_H_range(h, k, r_diff, min_H, max_H)

                if r_diff > 0:
                    r_larger = "r_2"
                elif r_diff < 0:
                    r_larger = "r_1"
                else:
                    r_larger = "equal"

                records.append({
                    "min_H": min_H,
                    "max_H": max_H,
                    "r_1_val": r_1_val,
                    "r_2_val": r_2_val,
                    "r_diff": r_diff,
                    "r_larger": r_larger,
                    "x_lower": x_lo,
                    "x_upper": x_hi,
                    "constraint": _iii_constraint_string(x_lo, x_hi),
                    "r_1": r_1,
                    "r_2": r_2,
                    "s_1": s_1,
                    "s_2": s_2,
                })

        lowers = [rec["x_lower"] for rec in records if rec["x_lower"] is not None]
        uppers = [rec["x_upper"] for rec in records if rec["x_upper"] is not None]
        by_j[j] = {
            "records": records,
            # take most permissive bounds
            "x_lower": min(lowers) if lowers else None,
            "x_upper": max(uppers) if uppers else None,
        }

    all_lowers = [
        data["x_lower"]
        for data in by_j.values()
        if data["x_lower"] is not None
    ]
    all_uppers = [
        data["x_upper"]
        for data in by_j.values()
        if data["x_upper"] is not None
    ]

    return {
        "x_lower": max(all_lowers) if all_lowers else None,
        "x_upper": min(all_uppers) if all_uppers else None,
        "by_j": by_j,
        "sign_h": sign_h,
        "sign_k": sign_k,
        "sigma": sigma,
    }

_HILFSATZ4_V_LO = RR(5) / 17
_HILFSATZ4_V_HI = RR(17) / 5

def _hilfsatz4_v_bounds_for_successor(successor):
    """Per-successor Hilfsatz 4 bounds: v(A) in ((5/17)/min|H|, (17/5)/max|H|)."""
    r_1 = [(0, *successor[0]), _ETA_EXTS[0]]
    r_2 = [(0, *successor[0]), _ETA_EXTS[1]]
    s_1 = [(0, *successor[1]), _ETA_EXTS[0]]
    s_2 = [(0, *successor[1]), _ETA_EXTS[1]]
    (min_H, _, _), (max_H, _, _) = min_max_H(
        cf_value(r_1), cf_value(r_2), cf_value(s_1), cf_value(s_2)
    )
    min_abs_H = min(abs(min_H), abs(max_H))
    max_abs_H = max(abs(min_H), abs(max_H))
    return {
        "successor": successor,
        "min_H": min_H,
        "max_H": max_H,
        "min_abs_H": min_abs_H,
        "max_abs_H": max_abs_H,
        "x_lower": _HILFSATZ4_V_LO / min_abs_H,
        "x_upper": _HILFSATZ4_V_HI / max_abs_H,
        "r_1": r_1,
        "r_2": r_2,
        "s_1": s_1,
        "s_2": s_2,
    }

def hilfsatz4_v_range(successors):
    """Return the v(A) range forced by Hilfsatz 4 admissibility across successors.

    For each successor j, Hilfsatz 4 gives
      v(~u |~ w) = |H(r_1, r_2, s_1, s_2; c_plus, c_minus)| * v(A)
    with r_i = [0; u + eta_i], s_i = [0; w + eta_i], hence
      v(A) in ((5/17) / min|H|, (17/5) / max|H|).
    The overall range is the intersection of these open intervals.
    """
    by_j = {}
    for j, successor in enumerate(successors):
        record = _hilfsatz4_v_bounds_for_successor(successor)
        record["j"] = j
        by_j[j] = record

    return {
        "x_lower": max(rec["x_lower"] for rec in by_j.values()),
        "x_upper": min(rec["x_upper"] for rec in by_j.values()),
        "by_j": by_j,
    }


In [ ]:
cases = [

# case 2.2 and 2.1 
# cases 2.1 (i) and (ii) and 2.2 (i), (ii) and (iii) seems not necessary. 
# I(~1|~) and I(~2|~2) are connected and other intervals are contained in them
# also, I(~2, 1|~3), I(~3|~2) and I(~2|~2) are connected and admissible when  1 <= v(A) <= 1.97964288362460
{
    "intervals": [
        ([2, 1], [3]),
        ([3], [2]),
        ([2], [2]),
        ([1], []),
    ],
    "hk": [(0, 0), (1, 1)],
},

# case 2.3
# cases 2.3 (ii) and (iii) seems not necessary. 
# I(~3|~2), I(~2, 1, 3|~3, 1), I(~2|~) I(~1|~) are connected and admissible when  1.95590293813816 <= v(A) <= 5.18658976638272
{
    "intervals": [
        ([3], [2]),
        ([2, 1, 3], [3, 1]),
        ([2], []),
        ([1], []),
    ],
    "hk": [(0, 0), (1, 1)],
},

# cases 2.4 and cases 2.5 (i)
# cases 2.4 (i), (ii), (iii), and (iv) seems not necesarry.
# I(~2|~1) and I(~1|~) seems enough
# also, cases 2.5 (ii) seems not necesarry.
{
    "intervals": [
        ([3], [1, 2]),
        ([2, 1, 3], [1, 3, 1]),
        ([2], [1]),
        ([1], []),
    ],
    "hk": [(0, 1), (1, 0)],
},

# cases 2.6 (i)
# cases 2.6 (ii), (iii), (iv), seems not necesarry.
{
    "intervals": [
        ([3], [1]),
        ([2], []),
        ([1], []),
    ],
    "hk": [(0, 1), (1, 0)],
},

]

for j in range(len(cases)):
    case = cases[j]
    successors = case["intervals"]
    h, k = case["hk"][0]
    if j == 0 or case["hk"][0] != cases[j - 1]["hk"][0]:
        print()
        print(f"(-1)^(h + k) = {(-1) ** (h + k)}:")
        print("-" * 100)

    sr_1, sr_2, sr_details = successor_sr(successors, h, k)
    # check (i)
    # this condition seems trivial. in this case, \tilde{s}_1^j ,\tilde{r}_1^j are something like [0;(3,1)] or [0;(2,1)]
    # check (ii)
    ii = check_ii(h, k, sr_1[0], sr_2[-1])
    print(f"check (ii) s: {ii['s']}, r: {ii['r']}, all: {all(ii.values())}")

    # calculate range of v(A) restricted by (iii)
    iii = iii_v_range(successors, h, k, sr_1=sr_1, sr_2=sr_2)
    print(
    f"overall v(A) range from iii: "
    f"({iii['x_lower']}, {iii['x_upper']})"
    )

    # check admissibility of the intervals by using Hilfsatz 4
    hilfsatz4 = hilfsatz4_v_range(successors)

    print(
        f"overall v(A) range from Hilfsatz 4: "
        f"({hilfsatz4['x_lower']}, {hilfsatz4['x_upper']})"
    )

    print(
        f"overall v(A) range from iii and Hilfsatz 4: "
        f"({_intersect_bound(iii['x_lower'], hilfsatz4['x_lower'], 'lower')}, "
        f"{_intersect_bound(iii['x_upper'], hilfsatz4['x_upper'], 'upper')})"
    )

'''
the above calculation shows that:
if (-1)^(h + k) = 1,
    if (1 >= ) 0.978235380188999 <= v(A) <= 1.97964288362460,
        ([2, 1], [3]),
        ([3], [2]),
        ([2], [2]),
        ([1], []),
        are admissible and connected.
    if 1.95590293813816 <= v(A) <= 5.18658976638272 (>= 17/5),
        ([3], [2]),
        ([2, 1, 3], [3, 1]),
        ([2], []),
        ([1], []),
        are admissible and connected.
    from above, there are a collection of admissible T-intervals,
    whose union is connected and containing ([1], []), ([3], [2]), and ([2, 1], [3]) (or ([2], [])).

if (-1)^(h + k) = -1,
    if (1 >= ) 0.978412226846568 <= v(A) <= 1.98750784542276,
        ([3], [1, 2]),
        ([2, 1, 3], [1, 3, 1]),
        ([2], [1]),
        ([1], []),
        are admissible and connected.
    if 1.77765186408743 <= v(A) <= 8.12741008384871 (>= 17/5),
        ([3], [1]),
        ([2], []),
        ([1], []),
        are admissible and connected.
    from above, there are a collection of admissible T-intervals,
    whose union is connected and containing ([1], []), ([3], [1, 2]) (or ([3], [1])), and ([2], [1]) (or ([2], [])).
'''



(-1)^(h + k) = 1:
----------------------------------------------------------------------------------------------------
check (ii) s: True, r: True, all: True
overall v(A) range from iii: (0.978235380188999, 1.97964288362460)
overall v(A) range from Hilfsatz 4: (0.808428126706646, 3.10088499093588)
overall v(A) range from iii and Hilfsatz 4: (0.978235380188999, 1.97964288362460)
check (ii) s: True, r: True, all: True
overall v(A) range from iii: (0.410380804651502, None)
overall v(A) range from Hilfsatz 4: (1.95590293813816, 5.18658976638272)
overall v(A) range from iii and Hilfsatz 4: (1.95590293813816, 5.18658976638272)

(-1)^(h + k) = -1:
----------------------------------------------------------------------------------------------------
check (ii) s: True, r: True, all: True
overall v(A) range from iii: (0.978412226846568, None)
overall v(A) range from Hilfsatz 4: (0.808428126706646, 1.98750784542276)
overall v(A) range from iii and Hilfsatz 4: (0.978412226846568, 1.98750784542276)

In [250]:
# initial point of the T-interval : [0;a_1, ... , a_k + r_1^j] + [0;a_{-1}, ... , a_{-h} + s_1^j]
# end point of the T-interval : [0;a_1, ... , a_k + r_2^j] + [0;a_{-1}, ... , a_{-h} + s_2^j]
#
# The H-inequality
#   (-1)^h * v(A) * (r_2 - r_1)
#     >= (-1)^k * (r_2 - r_1) * H(s_1, s_2, r_2, r_1, c_plus, c_minus)
# is equivalent to
#   (-1)^{h+k} ([0; a_1,...,a_k+r_1] + [0; a_{-1},...,a_{-h}+s_1])
#     <= (-1)^{h+k} ([0; a_1,...,a_k+r_2] + [0; a_{-1},...,a_{-h}+s_2]).

sr_1, sr_2, sr_details = successor_sr(successors, h, k)

for detail in sr_details:
    j = detail["j"]
    successor = detail["successor"]
    sr1_kept = detail["sr_1"]
    sr1_info = detail["sr_1_info"]
    sr2_kept = detail["sr_2"]
    sr2_info = detail["sr_2_info"]

    print(f"successor j={j}: {successor}")
    print(
        f"  sr_1 via H: a<=b={sr1_info['a_leq_b']}, b<=a={sr1_info['b_leq_a']}, "
        f"margins=({sr1_info['margin_a_b']}, {sr1_info['margin_b_a']}) "
        f"-> keep {len(sr1_kept)}"
    )
    for s, r in sr1_kept:
        print(f"    kept s={s}, r={r}")
    print(
        f"  sr_2 via H: a<=b={sr2_info['a_leq_b']}, b<=a={sr2_info['b_leq_a']}, "
        f"margins=({sr2_info['margin_a_b']}, {sr2_info['margin_b_a']}) "
        f"-> keep {len(sr2_kept)}"
    )
    for s, r in sr2_kept:
        print(f"    kept s={s}, r={r}")

successor j=0: ([2], [1])
  sr_1 via H: a<=b=False, b<=a=True, margins=(0.0466879096308383, -0.192437662600920) -> keep 1
    kept s=[(0, 2), (1, 2)], r=[(0, 1), (3, 1)]
  sr_2 via H: a<=b=False, b<=a=False, margins=(-0.0448573282176706, -0.00350283339720372) -> keep 2
    kept s=[(0, 2), (3, 1)], r=[(0, 1), (1, 2)]
    kept s=[(0, 2), (2, 1)], r=[(0, 1), (1, 3)]
successor j=1: ([1], [])
  sr_1 via H: a<=b=False, b<=a=True, margins=(0.0340676391187898, -0.182112832059178) -> keep 1
    kept s=[(0, 1), (1, 2)], r=[(0,), (1, 3)]
  sr_2 via H: a<=b=True, b<=a=False, margins=(-0.313384190933276, 0.0518596952215266) -> keep 1
    kept s=[(0, 1), (2, 1)], r=[(0,), (3, 1)]


In [187]:
# check (i)
# this condition seems trivial. in this case, \tilde{s}_1^j ,\tilde{r}_1^j are something like [0;(3,1)] or [0;(2,1)]

# check (ii)
ii = check_ii(h, k, sr_1[0], sr_2[-1])
print(f"check (ii) s: {ii['s']}, r: {ii['r']}, all: {all(ii.values())}")

# calculate range of v(A) restricted by (iii)
iii = iii_v_range(successors, h, k, sr_1=sr_1, sr_2=sr_2)

'''
# print records of (iii)

for j, data in iii["by_j"].items():
    print(f"j={j}")
    for rec in data["records"]:
        print(
            f"  min_H={rec['min_H']}, max_H={rec['max_H']}, "
            f"r_larger={rec['r_larger']} "
            f"(r_1={rec['r_1_val']}, r_2={rec['r_2_val']})"
        )
        print(
            f"    r_1={rec['r_1']}, s_1={rec['s_1']}, "
            f"r_2={rec['r_2']}, s_2={rec['s_2']}"
        )
        print(f"    => {rec['constraint']}")
    print(f"  at j={j}: x_lower={data['x_lower']}, x_upper={data['x_upper']}")

print(
    f"signs: (-1)^h={iii['sign_h']}, (-1)^k={iii['sign_k']}, "
    f"(-1)^(h+k)={iii['sigma']}"
)
'''

print(f"overall v(A) range from (iii): {iii['x_lower']} <= x <= {iii['x_upper']}")

# check admissibility of the intervals by using Hilfsatz 4
# r_i =  [0;u_1, ... , u_\nu + \eta_i]
# s_i = [0;w_1, ... , w_\mu + \eta_i]
# then, v(~u_1, ... , u_\nu |~ w_1, ... , w_\mu ) = |H(r_1, r_2, s_1, s_2, c_plus, c_minus)| * v(A)

hilfsatz4 = hilfsatz4_v_range(successors)

'''
# print records of checking admissibility

for j, data in hilfsatz4["by_j"].items():
    print(
        f"j={j}: v(A) in ({data['x_lower']}, {data['x_upper']}) "
        f"(min_abs_H={data['min_abs_H']}, max_abs_H={data['max_abs_H']})"
    )
'''
print(
    f"overall v(A) range from Hilfsatz 4: "
    f"({hilfsatz4['x_lower']}, {hilfsatz4['x_upper']})"
)


check (ii) s: True, r: True, all: True
overall v(A) range from (iii): 0.968997788022124 <= x <= None
overall v(A) range from Hilfsatz 4: (1.77765186408743, 5.18658976638272)


In [175]:
#below are the code for numerical experiment

In [336]:
# 具体的なAについてI(~|~)が連結か計算する
# successorの中の要素のそれぞれと、指定したA = ([1,2], [2,3])というような形のデータに関して、T_interval([1,2] + 要素の1番目, [2,3] + 要素の2番目)を計算する
# それで得られた区間の和集合が連結かどうかを判定する

# A = (negative_side, positive_side)
successors = cases[4]["intervals"]
#A = ([2,1], [3])
A = ([3, 3, 1, 1],[3, 3, 2])

t_intervals = []
for j, successor in enumerate(successors):
    negative_side = list(A[0]) + list(successor[0])
    positive_side = list(A[1]) + list(successor[1])
    left, left_expr, right, right_expr = T_interval(negative_side, positive_side)
    t_intervals.append({
        "j": j,
        "successor": successor,
        "negative_side": negative_side,
        "positive_side": positive_side,
        "left": left,
        "right": right,
        "left_expr": left_expr,
        "right_expr": right_expr,
    })
    print(
        f"j={j}, successor={successor}, "
        f"I({negative_side}|{positive_side}) = [{left}, {right}]"
    )

# 和集合の連結性: 左端でソートし、隣り合う成分が重なる／接するかを見る
ordered = sorted(range(len(t_intervals)), key=lambda i: (t_intervals[i]["left"], t_intervals[i]["right"]))
union_connected = True
gaps = []

if ordered:
    component_start = t_intervals[ordered[0]]["left"]
    component_end = t_intervals[ordered[0]]["right"]
    component_members = [ordered[0]]

    for idx in ordered[1:]:
        left = t_intervals[idx]["left"]
        right = t_intervals[idx]["right"]
        if left <= component_end:
            # 重なるまたは接する → 同じ連結成分に吸収
            component_end = max(component_end, right)
            component_members.append(idx)
        else:
            # ギャップあり
            union_connected = False
            gaps.append({
                "gap": (component_end, left),
                "left_component": list(component_members),
                "next_j": idx,
            })
            print(
                f"gap: ({component_end}, {left}) "
                f"between component {component_members} and j={idx}"
            )
            component_start = left
            component_end = right
            component_members = [idx]

    print(
        f"final component covering [{component_start}, {component_end}], "
        f"members={component_members}"
    )

print(f"union connected: {union_connected}")
if gaps:
    print(f"number of gaps: {len(gaps)}")


j=0, successor=([3], [1, 2]), I([3, 3, 1, 1, 3]|[3, 3, 2, 1, 2]) = [0.608079840277256, 0.608171622948698]
j=1, successor=([2], [1]), I([3, 3, 1, 1, 2]|[3, 3, 2, 1]) = [0.608156034512641, 0.608491458128836]
final component covering [0.608079840277256, 0.608491458128836], members=[0, 1]
union connected: True


In [333]:
print(T_ratio([3, 3, 1, 1],[3, 3, 2]))


s_1 = cf_value([(0,1),(1,2)])
r_1 = cf_value([(0,),(1,3)])

s_2 = cf_value([(0,2),(3,1)])
r_2 = cf_value([(0,1),(1,2)])

c_plus = cf_value([(0,3),()])
c_minus = cf_value([(0,1,2),()])

print(c_plus, c_minus)

print(H(s_1,s_2,r_2,r_1,c_plus,c_minus))

print(T_interval([2, 1, 2],[3, 1]))
print(T_interval([2, 1, 1],[3]))

1.11002651006001
0.333333333333333 0.666666666666667
-0.696079837185263
(0.629788019610412, (0, [2, 1, 2], (1, 2), [3, 1], (3, 1)), 0.651007544163419, (0, [2, 1, 2], (2, 1), [3, 1], (1, 3)))
(0.643416998763751, (0, [2, 1, 1], (1, 2), [3], (1, 3)), 0.694390213631401, (0, [2, 1, 1], (2, 1), [3], (3, 1)))


In [178]:
s_1 = cf_value([(0, 2), (3, 1)])
r_1 = cf_value([(0, 2), (2, 1)])

s_2 = cf_value([(0, 3), (1, 2)])
r_2 = cf_value([(0, 1, 1), (3, 1)])

c_plus = cf_value([(0, 2),()])
c_minus = cf_value([(0, 2),()])

print(c_plus)

print(H(s_1, s_2, r_2, r_1, 0.2, 0.2))

0.500000000000000
1.34746887399463
